<a href="https://colab.research.google.com/github/SivaKumarTammineni/Neural-Network-SMS-Text-Classifier/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# suppress warnings
import warnings
warnings.filterwarnings('ignore')

# install stable tensorflow
!pip uninstall -y tf-nightly keras-nightly tb-nightly -q
!pip install -q tensorflow tensorflow-datasets

# import libraries
import tensorflow as tf
from tensorflow import keras

import pandas as pd
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

# check tensorflow version
print("TensorFlow Version:", tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# load datasets
train_df = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

test_df = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

# encode labels
train_df['label'] = train_df['label'].map({
    'ham': 0,
    'spam': 1
})

test_df['label'] = test_df['label'].map({
    'ham': 0,
    'spam': 1
})

# extract messages and labels
train_messages = train_df['message'].astype(str).values
test_messages = test_df['message'].astype(str).values

train_labels = train_df['label'].values
test_labels = test_df['label'].values

# text vectorization
encoder = tf.keras.layers.TextVectorization(
    max_tokens=5000,
    output_sequence_length=100
)

encoder.adapt(train_messages)

# build model
model = tf.keras.Sequential([
    encoder,

    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=32
    ),

    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(32)
    ),

    tf.keras.layers.Dense(32, activation='relu'),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(1, activation='sigmoid')
])

# compile
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# train
history = model.fit(
    train_messages,
    train_labels,
    epochs=10,
    validation_data=(test_messages, test_labels),
    batch_size=32,
    verbose=1
)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
# function to predict messages
def predict_message(pred_text):

    # convert to tensor
    pred_input = tf.constant([str(pred_text)])

    # make prediction
    prediction = model(pred_input, training=False).numpy()[0][0]

    # classify
    label = "spam" if prediction >= 0.5 else "ham"

    return [float(prediction), label]


pred_text = "how are you doing today?"

prediction = predict_message(pred_text)

print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
